# S24 · A system that settles — Markov chains and the long run

We take a tiny weather system that hops between "sunny" and "rainy", write its
jump chances as a small table, and roll it forward day by day until the chances
stop changing. That settled mix, the long-run share of sunny days, is the one
number we usually want from a system like this. We find it two ways and watch
them agree.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- New to the idea of a "system that hops between states"? Open the primer
  `primers/markov_chains.md` for a ten-minute, picture-first version.
- New to tables of numbers (matrices)? Open `primers/vectors_and_matrices.md`.
- Already confident with code or with Markov chains? Skip ahead to the cell
  marked **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy and matplotlib.
# Google Colab already ships both, so there is nothing to install.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                 # arrays, tables of numbers, and random draws
import matplotlib.pyplot as plt    # drawing charts

# A seed fixes the "random" parts later so everyone gets the same result.
np.random.seed(0)

print("Libraries ready.")

## Step 1 — describe the system in plain words

Our system is the weather. Each day it is in exactly one of two **states**:
`Sunny` or `Rainy`. Tomorrow's weather depends only on today's, not on the whole
month before. That "only today matters" rule is what makes this a **Markov chain**.

The rule for changing is:

- If today is Sunny: 70% chance Sunny tomorrow, 30% chance Rainy.
- If today is Rainy: 40% chance Sunny tomorrow, 60% chance Rainy.

(A phone customer who is "loyal" or "ported to a rival" has the very same shape:
two states, and a chance of switching each month. Weather is just easier to picture.)

We give our two states names, in the order we will use everywhere below.

In [ ]:
# The names of our two states, in a fixed order.
state_names = ["Sunny", "Rainy"]

print("Our two states are:", state_names)

## Step 2 — write the jump chances as a table

We collect all four chances in one small square table `P`, called the
**transition matrix**.

- Row 0 is "today is Sunny", row 1 is "today is Rainy".
- Column 0 is "tomorrow Sunny", column 1 is "tomorrow Rainy".
- So `P[i, j]` is the chance of going **from** state i **to** state j.

Because from any state *something* must happen tomorrow, **each row must add up to
1**.

In [ ]:
# Build P. Rows are "from" (today), columns are "to" (tomorrow).
#                      to Sunny   to Rainy
P = np.array([[0.7,       0.3],      # from Sunny
              [0.4,       0.6]])     # from Rainy

print("Transition matrix P:")
print(P)

## Step 3 — check that each row adds up to 1

This is the one rule a transition matrix must obey. It is worth checking every
time, because a typo here quietly breaks everything later.

In [ ]:
# Add up each row (axis=1 means "sum across the columns of each row").
row_sums = P.sum(axis=1)

print("Sum of each row:", row_sums)
print("Both rows sum to 1, so P is a valid transition matrix.")

## Step 4 — start from a known day

We write today's weather as a small list of chances, one per state. Suppose we are
certain it is Sunny today. Then the list is `[1.0, 0.0]`: chance 1 of Sunny, chance
0 of Rainy.

In [ ]:
# A list of chances: 100% Sunny, 0% Rainy.
today = np.array([1.0, 0.0])

print("Today's chances:", today)
print("  chance of Sunny =", today[0], "  chance of Rainy =", today[1])

## Step 5 — take one step forward

To get tomorrow's chances, we combine today's chances with the jump table. We wrap
that one step in a small helper, `step_forward`, so the main line reads like plain
English: `tomorrow = step_forward(today, P)`. You do not need the mechanics to read
the result: it mixes today's chances with the jump rule to give tomorrow's.

In [ ]:
def step_forward(today_chances, jump_table):
    """Combine today's chances with the jump table to get tomorrow's chances."""
    return today_chances @ jump_table

# Advance one day by mixing today's chances with the jump table.
tomorrow = step_forward(today, P)

print("Tomorrow's chances:", tomorrow.round(3))
print("  chance of Sunny =", round(tomorrow[0], 3),
      "  chance of Rainy =", round(tomorrow[1], 3))
print("Starting from Sunny, there is now a 30% chance of Rain tomorrow.")

## Step 6 — keep stepping and watch it settle

If we apply the jump table again and again, we walk further into the future. We
record the chance of Sunny on each day so we can plot it. Watch the numbers stop
changing after a while: the system is **settling down**.

In [ ]:
number_of_days = 15

# We will record the chance of Sunny on each day.
prob_sunny_each_day = []

# Start fresh from "certainly Sunny".
distribution = np.array([1.0, 0.0])

for day in range(number_of_days):
    # Record the chance of Sunny today, before stepping.
    prob_sunny_each_day.append(distribution[0])
    # Step one day forward.
    distribution = step_forward(distribution, P)

# Print the chance of Sunny for each day.
for day in range(number_of_days):
    print("day", day, " chance of Sunny =", round(prob_sunny_each_day[day], 4))

## Step 7 — plot how it settles

The chance of Sunny starts at 1.0 and drifts down to a steady value. That steady
value is the **long-run share of sunny days**. This settled mix is the whole point:
it is what the weather "does on average" once it forgets today.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(range(number_of_days), prob_sunny_each_day, "-o", color="#2E75B6")
plt.xlabel("day number")
plt.ylabel("chance of Sunny")
plt.title("Starting from Sunny, the chance of Sunny settles down")
plt.ylim(0, 1.05)
plt.show()

## Step 8 — read off the settled mix

Once the numbers have stopped moving, the last day's chances *are* the long-run
mix. We call this settled mix the **stationary distribution** and write it `pi`.
(The name just means "the mix that stays put".) We take the distribution after all
those steps and print it.

In [ ]:
# After the loop above, `distribution` holds the chances after 15 steps,
# by which point they have essentially stopped moving.
settled_mix = distribution

print("Settled mix (long-run chances):", settled_mix.round(4))
print("  long-run chance of Sunny =", round(settled_mix[0], 4))
print("  long-run chance of Rainy =", round(settled_mix[1], 4))
print()
print("So in the long run about 57% of days are sunny, whatever today is.")

## Step 9 — confirm it by actually playing the weather

Here is a completely different check, and it needs no matrices at all. We *play*
the weather day by day: each morning we roll the dice using today's row of chances
to pick tomorrow. Do this for a long time and count the fraction of sunny days. It
should land on the same settled mix we just read off.

In [ ]:
number_of_days_to_simulate = 20000

# Start in state 0 (Sunny). We track the current state as 0 or 1.
current_state = 0
sunny_day_count = 0

for day in range(number_of_days_to_simulate):
    if current_state == 0:
        sunny_day_count = sunny_day_count + 1
    # Look up the row of P for the current state: the chances for tomorrow.
    chances_for_tomorrow = P[current_state]
    # Draw tomorrow's state at random using those chances.
    current_state = np.random.choice([0, 1], p=chances_for_tomorrow)

fraction_sunny = sunny_day_count / number_of_days_to_simulate

print("Played out over", number_of_days_to_simulate, "days:")
print("  simulated fraction of Sunny days :", round(fraction_sunny, 4))
print("  settled mix from stepping forward:", round(settled_mix[0], 4))
print()
print("They agree - the arithmetic and the simulation tell the same story.")

### Stretch (optional) — find the settled mix in one exact shot

Skip this if you are new to code or to matrices; you already have the answer two
ways. For anyone who has met **eigenvectors**, here is the elegant version. The
settled mix `pi` is exactly the mix that one more step leaves unchanged:
`pi @ P = pi`. Rearranged, that says `pi` is a **left eigenvector of `P` with
eigenvalue 1**. So we ask NumPy for the eigenvectors of `P` transposed, pick the
one whose eigenvalue is 1, and scale it so its entries add to 1.

For this particular `P` the exact answer is the fractions 4/7 and 3/7.

In [ ]:
# Eigen-decomposition of P transposed gives the LEFT eigenvectors of P.
eigenvalues, eigenvectors = np.linalg.eig(P.T)

# Find the eigenvector whose eigenvalue is closest to 1.
index_of_one = np.argmin(np.abs(eigenvalues - 1.0))
stationary = np.real(eigenvectors[:, index_of_one])

# An eigenvector can come out with any length or sign. A set of chances must be
# non-negative and add to 1, so we make it positive and divide by its total.
stationary = np.abs(stationary)
stationary = stationary / stationary.sum()

print("Stationary mix from the eigenvector:", stationary.round(4))
print("Exact fractions 4/7 and 3/7        :", round(4 / 7, 4), round(3 / 7, 4))
print("Settled mix from stepping forward  :", settled_mix.round(4))
print()
print("All three routes give the same long-run mix.")

## What you just did

You described a random system as two states with a table of jump chances (rows
adding to 1), stepped a starting mix forward until it stopped moving, and read off
the long-run mix, the **stationary distribution**. Then you confirmed it by simply
playing the weather at random for a long time. That settled mix is the one thing we
usually want from such a system, and you found it without any heavy machinery.

Next notebook: `02_inventory_by_simulation.ipynb`, where the "system that hops
between states" is a shop's stock level, and playing it out tells us whether a
restock rule is any good.